# HIL Calibration / Digital Twin Evidence

This notebook exercises SC-NeuroCore's local hardware-in-the-loop calibration, drift compensation, digital twin, SEU scrubbing, and telemetry comparison primitives.

## Evidence Boundary

This notebook uses generated protocols and synthetic telemetry only. It does not claim that a physical FPGA, ASIC, mixed-signal chip, or neuromorphic board has been calibrated, timed, or certified. Real hardware evidence requires captured device telemetry, bitstream provenance, clock/power conditions, and post-run calibration reports.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

from sc_neurocore.compiler.intelligence import (
    generate_digital_twin,
    generate_drift_compensator,
    generate_hil_calibration,
    ingest_telemetry,
    schedule_seu_scrubbing,
)

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent

equations = {
    "v": "v + dt * (i_syn - v / tau_m)",
    "u": "u + dt * (a * (b * v - u))",
}
parameter_ranges = {
    "tau_m": (5.0, 50.0),
    "i_syn_gain": (0.1, 2.0),
    "refractory_ms": (0.5, 8.0),
}
hil = generate_hil_calibration("sc_lif_hil", equations, parameters=parameter_ranges)
hil_summary = {
    "num_parameters": hil.num_parameters,
    "num_protocol_steps": len(hil.protocol_steps),
    "sweep_ranges": hil.sweep_ranges,
    "last_step": hil.protocol_steps[-1],
}
hil_summary

In [ ]:
drift = generate_drift_compensator(
    "sc_lif_hil",
    drift_rate_per_day=0.002,
    max_drift_tolerance=0.01,
    clock_freq_mhz=100,
    compensation_method="periodic_refresh",
)
refresh_match = re.search(r"localparam REFRESH_CYCLES = (\d+);", drift.verilog_controller)
assert refresh_match is not None
refresh_cycles = int(refresh_match.group(1))
expected_ms = 0.01 / 0.002 * 24 * 3600 * 1000

drift_summary = {
    "refresh_interval_ms": drift.refresh_interval_ms,
    "expected_refresh_interval_ms": expected_ms,
    "refresh_cycles": refresh_cycles,
    "method": drift.compensation_method,
    "has_verilog_module": "module sc_lif_hil_drift_ctrl" in drift.verilog_controller,
}
assert drift_summary["refresh_interval_ms"] == expected_ms
assert drift_summary["has_verilog_module"] is True
drift_summary

In [ ]:
twin_source = generate_digital_twin("sc_lif_hil", equations, "artix7_local_test")
namespace: dict[str, object] = {}
exec(compile(twin_source, "sc_lif_hil_twin.py", "exec"), namespace)
Twin = namespace["ScLifHilTwin"]
twin = Twin()
state0 = twin.step({"v": 0.25, "u": -0.05})
state1 = twin.step({"v": 0.28, "u": -0.04})
comparison = twin.compare({"v": 0.30, "u": -0.08})

digital_twin_summary = {
    "generated_class": "ScLifHilTwin",
    "cycle": twin.cycle,
    "state0": state0,
    "state1": state1,
    "comparison": comparison,
}
assert digital_twin_summary["cycle"] == 2
assert comparison["v"] > 0.0
digital_twin_summary

In [ ]:
leo = schedule_seu_scrubbing(2_000_000, orbit_altitude_km=400, shielding_mm_al=3.0)
higher_orbit = schedule_seu_scrubbing(2_000_000, orbit_altitude_km=1200, shielding_mm_al=3.0)
seu_summary = {
    "leo_interval_ms": leo.interval_ms,
    "higher_orbit_interval_ms": higher_orbit.interval_ms,
    "leo_frames_per_cycle": leo.frames_per_cycle,
    "higher_orbit_rate": higher_orbit.expected_seu_rate,
}
assert higher_orbit.interval_ms < leo.interval_ms
assert leo.frames_per_cycle == 2_000_000 // 1024
seu_summary

In [ ]:
healthy = ingest_telemetry(
    [{"v": 0.25, "u": -0.05}, {"v": 0.28, "u": -0.04}],
    [{"v": 0.25, "u": -0.05}, {"v": 0.281, "u": -0.041}],
    drift_threshold=0.01,
)
drifted = ingest_telemetry(
    [{"v": 0.25, "u": -0.05}, {"v": 0.90, "u": -0.04}],
    [{"v": 0.25, "u": -0.05}, {"v": 0.28, "u": -0.04}],
    drift_threshold=0.1,
)
telemetry_summary = {
    "healthy_samples": healthy.samples,
    "healthy": healthy.healthy,
    "drifted_healthy": drifted.healthy,
    "drifted_max_drift": drifted.max_drift,
    "drifted_alerts": drifted.alerts,
}
assert healthy.healthy is True
assert drifted.healthy is False
assert drifted.max_drift > 0.1
telemetry_summary

In [ ]:
manifest = {
    "schema_version": "sc-neurocore.hil-digital-twin-evidence.v1",
    "hil_summary": hil_summary,
    "drift_summary": drift_summary,
    "digital_twin_summary": digital_twin_summary,
    "seu_summary": seu_summary,
    "telemetry_summary": telemetry_summary,
    "evidence_boundary": "Generated local protocols and synthetic telemetry only; no physical hardware calibration or certification claim.",
}
manifest